<a href="https://colab.research.google.com/github/put-star/Flask_Blog1/blob/master/project%20ecommers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

folder_path = '/content/drive/MyDrive/data ecommers'

# Tampilkan semua file di folder
for file in os.listdir(folder_path):
    print(file)

final_global_economic_crisis_dataset.csv


In [6]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv('/content/drive/MyDrive/data economic GDP/final_global_economic_crisis_dataset.csv')
print(f"Shape awal: {df.shape}")

# ============================================================
# 2. INSPEKSI AWAL
# ============================================================
print("\n--- Info ---")
print(df.info())
print("\n--- Missing values ---")
print(df.isnull().sum())
print("\n--- Duplikat:", df.duplicated().sum())
print("\n--- Statistik ---")
print(df.describe().round(2))


Shape awal: (6347, 19)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6347 entries, 0 to 6346
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   country_name                 6347 non-null   object 
 1   country_code                 6347 non-null   object 
 2   year                         6347 non-null   int64  
 3   region                       6347 non-null   object 
 4   income_group                 6347 non-null   object 
 5   gdp_growth_annual_pct        6347 non-null   float64
 6   gdp_growth_lag1              6347 non-null   float64
 7   gdp_growth_3yr_avg           6347 non-null   float64
 8   is_recession                 6347 non-null   int64  
 9   target_recession_next_year   6347 non-null   float64
 10  inflation_rate_pct           6347 non-null   float64
 11  unemployment_rate_pct        6347 non-null   float64
 12  lending_interest_rate_pct    6347 non-n

FIX TIPE DATA

In [7]:
# target_recession_next_year masih float → ubah ke int
df['target_recession_next_year'] = df['target_recession_next_year'].astype(int)

# Pastikan year bertipe int
df['year'] = df['year'].astype(int)

# Kolom kategorikal → category (hemat memori)
cat_cols = ['country_name', 'country_code', 'region', 'income_group']
for col in cat_cols:
    df[col] = df[col].astype('category')

print("\n[OK] Tipe data diperbaiki")


[OK] Tipe data diperbaiki


# 5. TANGANI OUTLIER EKSTREM
# Strategi: WINSORIZE (clip) — lebih aman daripada hapus,
# karena ini data ekonomi yang memang bisa ekstrem

In [8]:

def winsorize_col(series, lower_pct=0.01, upper_pct=0.99):
    """Clip nilai di luar percentile lower-upper."""
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return series.clip(lower, upper)

# Kolom yang perlu diwinsorisasi
cols_to_winsorize = [
    'gdp_growth_annual_pct',
    'gdp_growth_lag1',
    'gdp_growth_3yr_avg',
    'inflation_rate_pct',
    'lending_interest_rate_pct',
    'real_interest_rate_pct',
    'trade_pct_gdp',
    'external_debt_pct_gni',
    'reserves_months_import',
    'current_account_balance_usd',
]

df_clean = df.copy()
for col in cols_to_winsorize:
    df_clean[col] = winsorize_col(df[col])
    print(f"  Winsorize {col}: [{df[col].min():.2f}, {df[col].max():.2f}]"
          f" → [{df_clean[col].min():.2f}, {df_clean[col].max():.2f}]")

  Winsorize gdp_growth_annual_pct: [-54.40, 149.97] → [-17.02, 19.53]
  Winsorize gdp_growth_lag1: [-64.05, 149.97] → [-17.45, 19.53]
  Winsorize gdp_growth_3yr_avg: [-33.45, 80.11] → [-11.05, 15.79]
  Winsorize inflation_rate_pct: [-42.92, 225690.06] → [-10.94, 513.85]
  Winsorize lending_interest_rate_pct: [0.50, 1443.61] → [2.69, 57.70]
  Winsorize real_interest_rate_pct: [-4582.66, 228.09] → [-40.77, 37.53]
  Winsorize trade_pct_gdp: [0.02, 863.20] → [22.19, 322.71]
  Winsorize external_debt_pct_gni: [0.24, 1343.27] → [2.62, 251.27]
  Winsorize reserves_months_import: [0.00, 79.24] → [0.13, 21.52]
  Winsorize current_account_balance_usd: [-993142000000.00, 443374257116.43] → [-73909819795.85, 108639941348.83]


6. VALIDASI RANGE LOGIS

In [9]:
# Unemployment tidak boleh negatif
df_clean['unemployment_rate_pct'] = df_clean['unemployment_rate_pct'].clip(lower=0)

# Population tidak boleh negatif
df_clean['population_total'] = df_clean['population_total'].clip(lower=0)

# is_recession & target hanya boleh 0 atau 1
assert df_clean['is_recession'].isin([0, 1]).all(), "is_recession ada nilai selain 0/1!"
assert df_clean['target_recession_next_year'].isin([0, 1]).all(), "target ada nilai selain 0/1!"

print("\n[OK] Validasi range logis selesai")


[OK] Validasi range logis selesai


7. SORT DATA (penting untuk analisis time series)

In [10]:
df_clean = df_clean.sort_values(['country_code', 'year']).reset_index(drop=True)
print("[OK] Data diurutkan per negara per tahun")

[OK] Data diurutkan per negara per tahun


8. CEK KONSISTENSI LOGIS

In [11]:
# Cek: is_recession=1 seharusnya gdp_growth negatif (tidak wajib, tapi perlu dicermati)
suspicious = df_clean[(df_clean['is_recession'] == 1) & (df_clean['gdp_growth_annual_pct'] > 5)]
print(f"\n[INFO] Baris resesi tapi GDP growth > 5%: {len(suspicious)} → perlu dicermati")
print(suspicious[['country_name', 'year', 'gdp_growth_annual_pct', 'is_recession']].head())


[INFO] Baris resesi tapi GDP growth > 5%: 0 → perlu dicermati
Empty DataFrame
Columns: [country_name, year, gdp_growth_annual_pct, is_recession]
Index: []


# 9. RINGKASAN AKHIR

In [12]:
print("\n" + "="*50)
print("RINGKASAN CLEANING")
print("="*50)
print(f"Shape akhir  : {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"Duplikat     : {df_clean.duplicated().sum()}")
print(f"Tahun        : {df_clean['year'].min()} - {df_clean['year'].max()}")
print(f"Negara       : {df_clean['country_code'].nunique()}")
print(f"Distribusi target:")
print(df_clean['target_recession_next_year'].value_counts())


RINGKASAN CLEANING
Shape akhir  : (6347, 19)
Missing values: 0
Duplikat     : 0
Tahun        : 1992 - 2022
Negara       : 214
Distribusi target:
target_recession_next_year
0    5279
1    1068
Name: count, dtype: int64
